### Objetivo

* Comprobar la homologación de los datos (ids iguales para los 3 archivos provistos)
* Realizar test para determinar si una pregunta es basal o no basal de forma estadística. Esto es encontrar algún umbral de valores nulos que indiquen que una pregunta se debería considerar basal o no basal. Para ello, se inicia con las preguntas que corresponden a la depresión, codificadas como las que inician con 'pmd'

In [2]:
import pandas as pd 
import matplotlib.pyplot as plt
import plotly.express as px

### Carga de los 3 datasets (Previamente innominados)

In [4]:
df_base_de_datos = pd.read_excel(r"C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\data\base datos proyecto infante juvenil 4 OCTUBRE 2010.xlsx")
df_sintomas_disc = pd.read_excel(r"C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\data\Sintomas_disc_Infante_Juvenil_n=1558.xlsx")
df_sub_umbrales  = pd.read_excel(r"C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\data\sub_umbrales_jovenes_infantes.xlsx")

### Revisión de tamaños de los datasets

Esto se hace principalmente para ver la integridad de los datos (asegurarse que tengan el mismo número de filas y que estas corresponden a los mismos sujetos)

In [6]:
print(df_base_de_datos.shape)
print(df_sintomas_disc.shape)
print(df_sub_umbrales.shape)

(1558, 830)
(1558, 3503)
(1558, 261)


Se definen las columnas de cada dataset

In [8]:
cols_base_de_datos = (df_base_de_datos.columns)
cols_sintomas_disc = (df_sintomas_disc.columns)
cols_sub_umbrales = (df_sub_umbrales.columns)

print(cols_base_de_datos)
print('-'*50)
print(cols_sintomas_disc)
print('-'*50)
print(cols_sub_umbrales)

Index(['inum', 'estrato', 'psu', 'peso_std_provincia_ajustado',
       'peso_ajustado_final', 'sex1', 'ed1', 'esc1', 'eciv1', 'act1',
       ...
       'NPERSON', 'Nin_Inf', 'ICD_1', 'dx_s1', 'ICD_2', 'ICD_3', 'ICDi_1',
       'ICDi_2', 'ICDi_3', 'grupox'],
      dtype='object', length=830)
--------------------------------------------------
Index(['id', 'huid', 'pad001', 'pad001a', 'pad001b', 'pad001c', 'pad001d',
       'pad002', 'pad002a', 'pad002b',
       ...
       'pwq019', 'pwq020', 'pwq021', 'pwq022', 'pwq022a', 'pwq023', 'pwq023a',
       'pwq024', 'pwq025', 'pwq026'],
      dtype='object', length=3503)
--------------------------------------------------
Index(['inum', 'ed1', 'padcrt1y', 'padcrt2y', 'padcrt1m', 'padcrt2m',
       'padcrt3y', 'padcrt3m', 'padsymp', 'patsymp',
       ...
       'pszsubtimb', 'pszsubtimc', 'pszsubtimd', 'yszcrit', 'yszsymp',
       'yszsubt', 'yszsubtima', 'yszsubtimb', 'yszsubtimc', 'yszsubtimd'],
      dtype='object', length=261)


Se definen los identificadores de cada dataset

In [10]:
print(df_base_de_datos['inum'])
print(df_sintomas_disc['id'])
print(df_sintomas_disc['huid'])
print(df_sub_umbrales['inum'])

0          1
1          2
2          3
3          4
4          5
        ... 
1553    1875
1554    1876
1555    1877
1556    1879
1557    1880
Name: inum, Length: 1558, dtype: int64
0          1
1          2
2          3
3          4
4          5
        ... 
1553    1875
1554    1876
1555    1877
1556    1879
1557    1880
Name: id, Length: 1558, dtype: int64
0        1
1        2
2        3
3        4
4        5
        ..
1553    75
1554    76
1555    77
1556    79
1557    80
Name: huid, Length: 1558, dtype: int64
0          1
1          2
2          3
3          4
4          5
        ... 
1553    1875
1554    1876
1555    1877
1556    1879
1557    1880
Name: inum, Length: 1558, dtype: int64


A simple vista, sí existe integridad en cuanto a ids homologados entre los datasets.

### Revisión de homologación

Se revisa si entre los 3 datasets, ID, INUM Y HUID son iguales

In [13]:
#Revisión de Integridad de identificadores

print(sorted(list(df_base_de_datos['inum'])) == sorted(list(df_sub_umbrales['inum'])))
print(sorted(list(df_base_de_datos['inum'])) == sorted(list(df_sintomas_disc['id'])))
print(sorted(list(df_base_de_datos['inum'])) == sorted(list(df_sintomas_disc['huid'])))

True
True
False


Huid no es igual a ID e INUM

Se comprueba además que huid ni siquiera contiene los mismos valores únicos; a diferencia de ID e INUM cuyos valores sí son úncos (valores únicos = número de filas = 1558)

In [16]:
#Revisión de valores únicos para id y huid

unique_id = (df_base_de_datos['inum'].unique())
unique_huid = (df_sintomas_disc['huid'].unique())

print(len(list(unique_id)))
print(len(list(unique_huid)))

1558
100


### Preguntas Basales y No Basales

Por ahora sólo nos interesa ver si las preguntas relacionadas a Depresión son basales o no. Para ello se revisa el dataset de síntomas

In [19]:
depression_cols_sintomas_disc = [i for i in cols_sintomas_disc if i.startswith('pmd')]
depression_cols_sub_umbrales = [i for i in cols_sub_umbrales if i.startswith('pmd')]

In [20]:
depression_df_sintomas_disc = df_sintomas_disc[list(depression_cols_sintomas_disc)]
depression_df_sub_umbrales = df_sub_umbrales[list(depression_cols_sub_umbrales)]

In [21]:
#Información general del set de preguntas sobre depresión.

display(depression_df_sintomas_disc.info())
display(depression_df_sintomas_disc.head())
display(depression_df_sintomas_disc.describe())
display(depression_df_sintomas_disc.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1558 entries, 0 to 1557
Columns: 240 entries, pmd001 to pmd060c
dtypes: float64(240)
memory usage: 2.9 MB


None

,pmd001,pmd001a,pmd001b,pmd001c,pmd001d,pmd001e,pmd001f,pmd002,pmd002a,pmd002b,...,pmd057,pmd057a,pmd058,pmd058a,pmd059,pmd059a,pmd060,pmd060a,pmd060b,pmd060c
0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,0.0,NaN,NaN,NaN,NaN,NaN,2.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2.0,0.0,NaN,NaN,NaN,NaN,NaN,2.0,2.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,pmd001,pmd001a,pmd001b,pmd001c,pmd001d,pmd001e,pmd001f,pmd002,pmd002a,pmd002b,...,pmd057,pmd057a,pmd058,pmd058a,pmd059,pmd059a,pmd060,pmd060a,pmd060b,pmd060c
count,1556.000000,667.000000,316.000000,316.000000,232.000000,162.000000,162.000000,1556.000000,352.000000,179.000000,...,7.000000,5.0,7.0,7.0,0.0,0.0,99.000000,49.000000,49.000000,13.000000
mean,0.857326,0.947526,1.297468,1.468354,1.396552,1.586420,0.814815,0.452442,1.042614,1.340782,...,1.285714,2.0,0.0,0.0,NaN,NaN,0.989899,1.346939,0.530612,1.384615
std,0.990088,0.999372,0.956246,0.884942,0.919997,1.025423,0.985751,0.837037,1.086515,0.942779,...,0.951190,1.0,0.0,0.0,NaN,NaN,1.005038,0.947607,0.892143,0.960769
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.0,0.0,0.0,NaN,NaN,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,...,0.500000,1.0,0.0,0.0,NaN,NaN,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,2.000000,2.000000,2.000000,2.000000,0.000000,0.000000,2.000000,2.000000,...,2.000000,2.0,0.0,0.0,NaN,NaN,0.000000,2.000000,0.000000,2.000000
75%,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,0.000000,2.000000,2.000000,...,2.000000,3.0,0.0,0.0,NaN,NaN,2.000000,2.000000,2.000000,2.000000
max,2.000000,2.000000,2.000000,2.000000,2.000000,9.000000,2.000000,2.000000,9.000000,2.000000,...,2.000000,3.0,0.0,0.0,NaN,NaN,2.000000,2.000000,2.000000,2.000000


pmd001        2
pmd001a     891
pmd001b    1242
pmd001c    1242
pmd001d    1326
           ... 
pmd059a    1558
pmd060     1459
pmd060a    1509
pmd060b    1509
pmd060c    1545
Length: 240, dtype: int64

### Conteo de nulos por pregunta

Me interesa trabajar con este subconjunto de preguntas porque están ya identificadas en varios archivos y puedo comprobar 'manualmente' ciertos aspectos relacionados a valores perdidos.

In [23]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

display(depression_df_sintomas_disc.isnull().sum())

pmd001         2
pmd001a      891
pmd001b     1242
pmd001c     1242
pmd001d     1326
pmd001e     1396
pmd001f     1396
pmd002         2
pmd002a     1206
pmd002b     1379
pmd002c     1438
pmd003         2
pmd003a      802
pmd003b     1277
pmd003c     1277
pmd003d     1367
pmd003e     1434
pmdn01       477
pmdn02         2
pmd004         2
pmd004a     1207
pmd004b     1446
pmd004c     1286
pmd004d     1474
pmd004e     1286
pmd005         2
pmd005a     1445
pmd005b     1480
pmd005c     1163
pmd006         2
pmd006a     1259
pmd006b     1466
pmd006c     1259
pmd007         2
pmd007a     1435
pmd007b     1504
pmd007c     1118
pmdn03a        2
pmdn03b        2
pmd008         2
pmd008a     1027
pmd008b     1384
pmd008c     1436
pmd008d     1027
pmd009         2
pmd009a     1442
pmd009b     1489
pmd009c     1143
pmdn04a        2
pmdn04b        2
pmd010         2
pmd010a      630
pmd010b     1465
pmd010c     1491
pmd010d     1317
pmd011         2
pmd011a      739
pmd011b     1169
pmd011c     14

Es sencillo notar que las preguntas no terminadas en letras son consideradas basales dentro del DISC y coinciden con los valores perdidos por el sistema según el archivo 'Libro codigos Sintomas Infante Juvenil N=1558.doc' provisto. 

Por ejemplo, la pregunta 35 o pmd035, que en el cuestionario 'Dd-21.wpd' es 'Durante el último año–es decir, desde [[MENCIONE SUCESO]/[MENCIONE MES ACTUAL] del año pasado] – ¿hubo una  época en que te sentiste triste o deprimido(a) gran parte del tiempo (muchas veces)?' y en el libro de código es 'Seemed sad or depressed a lot of the time in past year', coincide en código y valores nulos perdidos por el sistema (87) entre el libro de código y los datos de 'Sintomas_disc_Infante_Juvenil_n=1558.xlsx'. Así mismo, hay conexión en cuanto a intención y sintaxis entre el libro de código en inglés y el cuestionario en español, así que es fácil ver que es efectivamente una pregunta que abre hacia otras 5 preguntas no basales (pmd035a-f).

Si esto es cierto para el resto de preguntas, y ya existen cantidades documentadas de valores perdidos, analiizar si una pregunta es basal o no basal a partir de umbrales estadísticos no es del todo válida, ya que existen preguntas cuyo porcentaje de pérdida por el sistema es muy alta, por ejemplo, la pregunta pmd056 ('Being sad/depressed/grouchy/irritable made teachers/bosses annoyed') tiene 1551 valores perdidos (nulos) y sería marcada como no basal.

De todas formas, este Notebook comprueba que existe homologación entre los datos de los 3 datasets